In [33]:
import os

# 1. 填入你解压出来的文件夹名字
project_folder_name = 'Project_Folder'  # <--- 请根据第一步看到的打印结果修改这里的名字！

# 2. 检查一下是否存在，如果存在就进入
if os.path.exists(project_folder_name):
    os.chdir(project_folder_name)
    print(f"成功进入文件夹: {os.getcwd()}")
else:
    print(f"没找到 {project_folder_name}，请检查第一步打印的文件列表。")

# 3. 再次确认 utils 是否在眼前
if 'utils' in os.listdir():
    print("太棒了！找到了 utils 文件夹，现在可以运行 import 了。")
else:
    print("还是没找到 utils，请检查你的压缩包结构。")

没找到 Project_Folder，请检查第一步打印的文件列表。
太棒了！找到了 utils 文件夹，现在可以运行 import 了。


In [34]:
import os
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from models.network_dncnn import DnCNN as net
from utils import utils_image as util
from utils import utils_model


In [35]:
class Config:
    def __init__(self):
        self.model_name = 'dncnn_25'
        self.noise_level_img = 25
        self.model_pool = 'model_zoo'
        self.results = 'results'

args = Config()
SIGMA = args.noise_level_img


In [36]:
IMG_PATH = "Real/bycicle.jpg"




if not os.path.exists(IMG_PATH):
    raise FileNotFoundError(f"Bicycle image not found: {IMG_PATH}")

print(f"Using Bicycle image: {IMG_PATH}")


Using Bicycle image: Real/bycicle.jpg


In [37]:
def load_bicycle_gray(path, target_hw=None):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"Cannot read image: {path}")
    if target_hw is not None:
        img = cv2.resize(img, (target_hw[1], target_hw[0]),
                         interpolation=cv2.INTER_CUBIC)
    return img



In [38]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on: {device}")

model_path = os.path.join(args.model_pool, args.model_name + '.pth')
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}")

model = net(in_nc=1, out_nc=1, nc=64, nb=17, act_mode='R')
model.load_state_dict(torch.load(model_path, map_location=device), strict=True)
model.eval()
for _, v in model.named_parameters():
    v.requires_grad = False
model = model.to(device)

print("DnCNN model loaded.")


Running on: cuda
DnCNN model loaded.


In [39]:
img_gt_sq = visual_data['clean']
img_noisy_sq = visual_data['noisy']
img_denoised_sq = visual_data['denoised']


psnr_noisy = util.calculate_psnr(img_gt_sq, img_noisy_sq, border=0)
psnr_denoised = util.calculate_psnr(img_gt_sq, img_denoised_sq, border=0)
ssim_denoised = util.calculate_ssim(img_gt_sq, img_denoised_sq, border=0)

print(f"Noisy PSNR:    {psnr_noisy:.2f} dB")
print(f"DnCNN PSNR:    {psnr_denoised:.2f} dB")
print(f"DnCNN SSIM:    {ssim_denoised:.4f}")


NameError: name 'visual_data' is not defined

In [ ]:
plt.figure(figsize=(15, 5))

# Column 1: Ground Truth
plt.subplot(1, 3, 1)
plt.imshow(img_gt_sq, cmap='gray')
plt.title("Ground Truth (Target)", fontsize=12)
plt.axis('off')

# Column 2: Noisy Input
plt.subplot(1, 3, 2)
plt.imshow(img_noisy_sq, cmap='gray')
plt.title(f"Noisy Input (σ={SIGMA})\nPSNR: {psnr_noisy:.2f}dB", fontsize=12)
plt.axis('off')

# Column 3: DnCNN Result
plt.subplot(1, 3, 3)
plt.imshow(img_denoised_sq, cmap='gray')
plt.title(
    f"DnCNN Result (σ={SIGMA})\n"
    f"PSNR: {psnr_denoised:.2f}dB / SSIM: {ssim_denoised:.3f}",
    fontsize=12,
    fontweight='bold'
)
plt.axis('off')

plt.tight_layout()
plt.savefig("dncnn_bicycle_combined_results.png", dpi=200)
plt.show()
